# 01 — Data Setup & Validation
**SVAMITVA Hackathon — AI-Based Feature Extraction from Drone Images**

This notebook:
1. Mounts Google Drive & installs dependencies
2. Copies data from Drive to Colab local SSD
3. Validates all orthophotos and shapefiles
4. Reports what's working and what's not

In [ ]:
# ── Cell 1: Environment Detection & Drive Mount ──────────────────
import os, sys, shutil
from pathlib import Path

IS_COLAB = 'google.colab' in sys.modules

if IS_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_ROOT = Path('/content/drive/MyDrive/IITT_AIML')
    LOCAL_ROOT = Path('/content/IITT_AIML')
else:
    DRIVE_ROOT = Path.home() / 'IITT_AIML'
    LOCAL_ROOT = DRIVE_ROOT

print(f'Running on: {"Colab" if IS_COLAB else "Local"}')
print(f'Drive root: {DRIVE_ROOT}')
print(f'Local root: {LOCAL_ROOT}')

In [ ]:
# ── Cell 2: Install Dependencies ─────────────────────────────────
if IS_COLAB:
    !pip install -q rasterio geopandas fiona shapely pyproj \
        segmentation-models-pytorch transformers \
        albumentations>=1.3.0 timm \
        tensorboard rioxarray
    # GPU info
    !nvidia-smi -L

In [ ]:
# ── Cell 3: Project Structure Setup ──────────────────────────────
dirs = [
    LOCAL_ROOT / 'imagery' / 'CG_train',
    LOCAL_ROOT / 'imagery' / 'PB_train',
    LOCAL_ROOT / 'imagery' / 'CG_test',
    LOCAL_ROOT / 'imagery' / 'PB_test',
    LOCAL_ROOT / 'tiles' / 'images',
    LOCAL_ROOT / 'tiles' / 'masks',
    LOCAL_ROOT / 'checkpoints',
    LOCAL_ROOT / 'outputs',
    LOCAL_ROOT / 'logs',
]
for d in dirs:
    d.mkdir(parents=True, exist_ok=True)

# Copy src/ from Drive to local (Colab only)
if IS_COLAB:
    src_drive = DRIVE_ROOT / 'src'
    src_local = LOCAL_ROOT / 'src'
    if src_drive.exists():
        if src_local.exists():
            shutil.rmtree(src_local)
        shutil.copytree(src_drive, src_local)
        print(f'src/ copied ({len(list(src_local.glob("*.py")))} modules)')
    else:
        raise FileNotFoundError(
            f'Upload src/ folder to Google Drive at: {src_drive}'
        )

sys.path.insert(0, str(LOCAL_ROOT))
print(f'Project root: {LOCAL_ROOT}')

In [ ]:
# ── Cell 4: Find & Extract Data Zips ─────────────────────────────
import zipfile

def extract_if_zip(zip_path, dest_dir):
    zip_path = Path(zip_path)
    if not zip_path.exists():
        return []
    print(f'  Extracting {zip_path.name} -> {dest_dir}')
    Path(dest_dir).mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(zip_path, 'r') as zf:
        zf.extractall(dest_dir)
    tifs = list(Path(dest_dir).rglob('*.tif'))
    shps = list(Path(dest_dir).rglob('*.shp'))
    return tifs, shps

def find_orthos(search_dir, extensions=('.tif', '.tiff', '.ecw')):
    found = []
    search_dir = Path(search_dir)
    if not search_dir.exists():
        return found
    for ext in extensions:
        found.extend(search_dir.rglob(f'*{ext}'))
    return sorted(found)

# ── Zip extraction rules ──
# Exact zip names from hackathon data:
#   CG_Training_dataSet_2.zip → CG orthophotos (BADETUMNAR, KUTRU)
#   CG_Training_dataSet_3.zip → CG orthophotos (MURDANDA, NAGUL, SAMLUR)
#   CG_shp-file.zip           → CG shapefiles
#   PB_training_dataSet_shp_file.zip → PB orthophotos + PB shapefiles

zip_extraction_rules = [
    # (pattern_in_name, destination, description)
    ('cg_training_dataset_2', LOCAL_ROOT / 'imagery' / 'CG_train', 'CG orthos batch 2'),
    ('cg_training_dataset_3', LOCAL_ROOT / 'imagery' / 'CG_train', 'CG orthos batch 3'),
    ('cg_training_dataset',   LOCAL_ROOT / 'imagery' / 'CG_train', 'CG orthos'),
    ('cg_shp',                LOCAL_ROOT / 'shp-file',              'CG shapefiles'),
    ('pb_training',           LOCAL_ROOT / 'imagery' / 'PB_train', 'PB data'),
    ('pb_train',              LOCAL_ROOT / 'imagery' / 'PB_train', 'PB data'),
]

if IS_COLAB:
    # Search for zips in multiple locations
    zip_search_dirs = [
        DRIVE_ROOT / 'zips',
        DRIVE_ROOT / 'imagery',
        DRIVE_ROOT / 'data',
        DRIVE_ROOT,
    ]
    
    all_zips = set()
    for search_dir in zip_search_dirs:
        if search_dir.exists():
            all_zips.update(search_dir.glob('*.zip'))
    
    print(f'Found {len(all_zips)} zip files:')
    for zf in sorted(all_zips):
        print(f'  {zf.name} ({zf.stat().st_size/1e9:.2f} GB)')
    
    # Extract each zip to the right place
    for zf in sorted(all_zips):
        name_lower = zf.stem.lower().replace('-', '_').replace(' ', '_')
        matched = False
        for pattern, dest, desc in zip_extraction_rules:
            if pattern in name_lower:
                print(f'\n  {zf.name} -> {desc} ({dest})')
                extract_if_zip(zf, dest)
                matched = True
                break
        if not matched:
            print(f'\n  {zf.name} -> unknown, extracting to imagery/')
            extract_if_zip(zf, LOCAL_ROOT / 'imagery')
    
    # Copy any non-zip orthophotos from Drive
    for state in ['CG_train', 'PB_train', 'CG_test', 'PB_test']:
        drive_dir = DRIVE_ROOT / 'imagery' / state
        local_dir = LOCAL_ROOT / 'imagery' / state
        if drive_dir.exists():
            for f in find_orthos(drive_dir):
                dest = local_dir / f.name
                if not dest.exists():
                    sz = f.stat().st_size / 1e9
                    print(f'  Copying {f.name} ({sz:.1f} GB)')
                    shutil.copy2(f, dest)

# Show what we found
print('\n' + '=' * 60)
print('ORTHOPHOTO INVENTORY')
print('=' * 60)
for state in ['CG_train', 'PB_train', 'CG_test', 'PB_test']:
    local_dir = LOCAL_ROOT / 'imagery' / state
    if local_dir.exists():
        orthos = find_orthos(local_dir)
        print(f'\n{state}: {len(orthos)} files')
        for f in orthos:
            print(f'  {f.name} ({f.stat().st_size/1e9:.2f} GB)')
    else:
        print(f'\n{state}: (not found)')

# Check for stray orthos
stray = find_orthos(LOCAL_ROOT / 'imagery')
known = set()
for state in ['CG_train', 'PB_train', 'CG_test', 'PB_test']:
    d = LOCAL_ROOT / 'imagery' / state
    if d.exists():
        known.update(find_orthos(d))
stray_files = [f for f in stray if f not in known]
if stray_files:
    print(f'\nStray orthos (may need manual sorting):')
    for f in stray_files:
        print(f'  {f.relative_to(LOCAL_ROOT)} ({f.stat().st_size/1e9:.2f} GB)')

In [ ]:
# ── Cell 5: Find & Copy Shapefiles ───────────────────────────────
def find_shapefiles(search_dir):
    return sorted(Path(search_dir).rglob('*.shp')) if Path(search_dir).exists() else []

# Copy CG shapefiles from Drive
if IS_COLAB:
    cg_shp_drive = DRIVE_ROOT / 'shp-file'
    cg_shp_local = LOCAL_ROOT / 'shp-file'
    if cg_shp_drive.exists() and not cg_shp_local.exists():
        shutil.copytree(cg_shp_drive, cg_shp_local)
        print(f'Copied CG shapefiles from Drive ({len(find_shapefiles(cg_shp_local))} files)')
    elif cg_shp_drive.exists():
        print(f'CG shapefiles already local ({len(find_shapefiles(cg_shp_local))} files)')

# PB shapefiles may have been extracted from the PB zip into PB_train/
# Search for them in various locations
pb_shp_local = LOCAL_ROOT / 'imagery' / 'PB_train' / 'shp-file'
if not pb_shp_local.exists() or not find_shapefiles(pb_shp_local):
    # Look for shapefiles anywhere inside PB_train extraction
    pb_train = LOCAL_ROOT / 'imagery' / 'PB_train'
    if pb_train.exists():
        all_pb_shps = find_shapefiles(pb_train)
        if all_pb_shps and not pb_shp_local.exists():
            # Shapefiles might be in a subfolder — find their parent
            shp_parents = set(f.parent for f in all_pb_shps)
            if len(shp_parents) == 1 and list(shp_parents)[0] != pb_shp_local:
                # All in one folder, create symlink or copy
                actual_dir = list(shp_parents)[0]
                if actual_dir != pb_shp_local:
                    pb_shp_local.mkdir(parents=True, exist_ok=True)
                    for f in actual_dir.iterdir():
                        dest = pb_shp_local / f.name
                        if not dest.exists():
                            shutil.copy2(f, dest)
                    print(f'Moved PB shapefiles to {pb_shp_local}')

shp_locations = {
    'CG': [
        LOCAL_ROOT / 'shp-file',
        DRIVE_ROOT / 'shp-file',
        LOCAL_ROOT / 'imagery' / 'CG_train' / 'shp-file',
    ],
    'PB': [
        LOCAL_ROOT / 'imagery' / 'PB_train' / 'shp-file',
        DRIVE_ROOT / 'imagery' / 'PB_train' / 'shp-file',
        # Also check if shapefiles are loose inside PB_train
        LOCAL_ROOT / 'imagery' / 'PB_train',
    ],
}

print('-- Shapefile inventory --')
for state, locs in shp_locations.items():
    found = []
    for loc in locs:
        found.extend(find_shapefiles(loc))
    found = list(set(found))
    print(f'{state}: {len(found)} shapefiles')
    for f in sorted(found):
        print(f'  {f.stem}')

In [ ]:
# ── Cell 6: Validate Orthophotos ─────────────────────────────────
import rasterio
import numpy as np

def validate_orthophoto(path):
    path = Path(path)
    try:
        with rasterio.open(path) as src:
            win = rasterio.windows.Window(0, 0, min(256, src.width), min(256, src.height))
            patch = src.read(window=win)
            return {
                'name': path.name, 'status': 'OK',
                'size_gb': path.stat().st_size / 1e9,
                'width': src.width, 'height': src.height,
                'bands': src.count, 'dtype': str(src.dtypes[0]),
                'crs': str(src.crs),
                'res_cm': round(abs(src.res[0]) * 100, 2),
            }
    except Exception as e:
        return {'name': path.name, 'status': f'ERROR: {e}',
                'size_gb': path.stat().st_size / 1e9}

print('=' * 80)
print('ORTHOPHOTO VALIDATION')
print('=' * 80)

working_orthos = []
failed_orthos = []

for state in ['CG_train', 'PB_train', 'CG_test', 'PB_test']:
    local_dir = LOCAL_ROOT / 'imagery' / state
    orthos = find_orthos(local_dir)
    if not orthos:
        continue
    print(f'\n-- {state} ({len(orthos)} files) --')
    for f in orthos:
        info = validate_orthophoto(f)
        if info['status'] == 'OK':
            print(f"  OK   {info['name']:45s} {info['width']}x{info['height']}  "
                  f"{info['bands']}bands  {info['crs']}  {info['res_cm']}cm/px")
            working_orthos.append((str(f), info, state))
        else:
            print(f"  FAIL {info['name']:45s} {info['status']}")
            failed_orthos.append((str(f), info, state))

print(f'\nWorking: {len(working_orthos)} | Failed: {len(failed_orthos)}')

In [ ]:
# ── Cell 7: Validate Shapefiles ──────────────────────────────────
import geopandas as gpd

print('=' * 80)
print('SHAPEFILE VALIDATION')
print('=' * 80)

for state, locs in shp_locations.items():
    shps = []
    for loc in locs:
        shps.extend(find_shapefiles(loc))
    shps = list(set(shps))
    if not shps:
        print(f'\n{state}: No shapefiles found')
        continue
    print(f'\n-- {state} ({len(shps)} shapefiles) --')
    for f in sorted(shps, key=lambda x: x.stem):
        try:
            gdf = gpd.read_file(f)
            n_total = len(gdf)
            gdf = gdf[gdf.geometry.notna()]
            gdf = gdf[gdf.geometry.is_valid]
            n_valid = len(gdf)
            print(f'  {f.stem:25s}  {n_valid:5d}/{n_total:5d} valid  '
                  f'{str(gdf.crs):20s}  {gdf.geom_type.unique().tolist()}')
        except Exception as e:
            print(f'  {f.stem:25s}  ERROR: {e}')

In [ ]:
# ── Cell 8: Building & Roof-Type Distribution ────────────────────
for state, locs in shp_locations.items():
    bldg_shp = None
    for loc in locs:
        for pattern in ['Built_Up_Area_type.shp', 'Built_Up_Area_typ.shp']:
            cand = loc / pattern
            if cand.exists():
                bldg_shp = cand
                break
        if bldg_shp:
            break
    if bldg_shp is None:
        print(f'{state}: No building shapefile found'); continue
    
    gdf = gpd.read_file(bldg_shp)
    gdf = gdf[gdf.geometry.notna() & gdf.geometry.is_valid]
    print(f'\n-- {state} Buildings ({len(gdf)}) --')
    
    roof_col = next((c for c in gdf.columns if 'roof' in c.lower()), None)
    if roof_col:
        roof_map = {1: 'RCC', 2: 'Tiled', 3: 'Tin/Metal', 4: 'Thatched'}
        for val, cnt in gdf[roof_col].value_counts().sort_index().items():
            pct = cnt / len(gdf) * 100
            print(f'  Roof {int(val)} ({roof_map.get(int(val),"??"):12s}): {cnt:6d} ({pct:5.1f}%)')

In [ ]:
# ── Cell 9: Save Data Manifest ───────────────────────────────────
import json

# Determine which shapefile directory to use for each state
shp_dirs = {}
for state, locs in shp_locations.items():
    for loc in locs:
        if loc.exists() and find_shapefiles(loc):
            shp_dirs[state] = str(loc)
            break

manifest = {
    'working_orthos': [
        {'path': p, 'state': s, 'name': info['name'],
         'crs': info.get('crs',''), 'res_cm': info.get('res_cm',0),
         'width': info.get('width',0), 'height': info.get('height',0),
         'bands': info.get('bands',0)}
        for p, info, s in working_orthos
    ],
    'failed_orthos': [
        {'path': p, 'state': s, 'error': info['status']}
        for p, info, s in failed_orthos
    ],
    'shapefile_dirs': shp_dirs,
}

manifest_path = LOCAL_ROOT / 'data_manifest.json'
with open(manifest_path, 'w') as f:
    json.dump(manifest, f, indent=2)

if IS_COLAB:
    shutil.copy2(manifest_path, DRIVE_ROOT / 'data_manifest.json')

print(f'Manifest saved: {manifest_path}')
print(f'  {len(manifest["working_orthos"])} working orthophotos')
print(f'  {len(manifest["failed_orthos"])} failed orthophotos')
print(f'\nProceed to 02_preprocess.ipynb')